# Unidad 4 · Cuaderno 01 · Verificación y validación

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2

**Unidad 4.** Validación, interpretación y comunicación de resultados
· **Subtema del plan 4.1**

Este cuaderno ejecuta lo que el libro expone en las secciones 4.1, 4.2 y 4.3. El texto no
repite la teoría, remite a ella por número de definición, de teorema, de
ejemplo, de listado, de tabla o de figura, y se ocupa de reproducir los
resultados publicados y de verificarlos.

**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad4/U4_01_verificacion_y_validacion.ipynb)

## Objetivos de aprendizaje

1. Distinguir verificación de validación y aplicar el orden que la Definición 4.1 del libro impone entre ambas.
2. Estimar el orden observado de un esquema numérico y reconocer por su valor un defecto de frontera, según el Teorema 4.1.
3. Construir una solución fabricada con álgebra simbólica y verificar con ella un código sin solución analítica, según la Definición 4.2.
4. Calibrar un modelo no lineal y reportar la precisión de sus parámetros con el Teorema 4.2 y con remuestreo.
5. Diagnosticar identificabilidad práctica y comparar modelos con el criterio de información corregido de la Ecuación 4.10.
6. Validar contra datos independientes con criterio declarado de antemano y examinar la estructura de los residuales.

## Puesta a punto

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("Entorno listo. Colab:", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
})

# Bandera de los ejercicios guiados. En la versión de trabajo vale False
# para que el cuaderno corra completo aunque falten celdas por resolver.
REVISAR = False


def verificar(nombre: str, obtenido, esperado: float,
              tol: float = 1.0e-3) -> bool:
    """Compara un resultado con el valor esperado sin detener el cuaderno."""
    if obtenido is None or (isinstance(obtenido, float) and np.isnan(obtenido)):
        print(f"[pendiente] {nombre}, la celda marcada COMPLETE sigue sin resolver")
        return False
    escala = abs(esperado) if esperado != 0.0 else 1.0
    error = abs(float(obtenido) - esperado) / escala
    estado = "ok" if error <= tol else "revisar"
    print(f"[{estado}] {nombre}, obtenido {float(obtenido):.6g}, "
          f"esperado {esperado:.6g}, error relativo {error:.2e}")
    if REVISAR:
        assert error <= tol, f"{nombre} no coincide con el valor esperado"
    return error <= tol


print("Semilla del curso:", SEMILLA)

In [ ]:
# Acceso a datos/ que funciona en Colab y en local, sin rutas absolutas.
# Si la carpeta no viaja con el cuaderno, las series se reconstruyen con la
# semilla del curso y con las cifras que el libro publica.

def carpeta_datos() -> Path:
    """Ubica datos/ subiendo por el árbol, o la crea junto al cuaderno."""
    base = Path.cwd()
    for nivel in [base, *base.parents][:4]:
        for candidata in (nivel / "datos", nivel / "03_cuadernos" / "datos"):
            if candidata.is_dir():
                return candidata
    destino = base / "datos"
    destino.mkdir(parents=True, exist_ok=True)
    return destino


def _cinetica_monod() -> pd.DataFrame:
    return pd.DataFrame({
        "S_g_L": [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 18.0, 25.0, 35.0],
        "mu_1_h": [0.0702, 0.1248, 0.1919, 0.2496, 0.2761,
                   0.3205, 0.3689, 0.3709, 0.3770, 0.3773]})


def _secado_calibracion() -> pd.DataFrame:
    t = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0,
                  4.0, 5.0, 6.0, 7.0, 8.0])
    gen = np.random.default_rng(SEMILLA)
    mr = np.round(np.exp(-0.350 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _secado_validacion() -> pd.DataFrame:
    t = np.array([0.5, 1.0, 1.5, 2.0, 2.75, 3.5, 4.5, 5.5, 6.5, 8.0])
    gen = np.random.default_rng(SEMILLA + 1)
    mr = np.round(np.exp(-0.362 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _caudal_mensual() -> pd.DataFrame:
    obs = [6.21, 5.01, 4.30, 9.39, 14.85, 18.28, 16.85, 17.12, 21.51, 22.75,
           19.33, 12.13, 7.65, 6.14, 4.54, 8.58, 14.73, 21.82, 14.36, 17.57,
           24.54, 31.26, 21.30, 11.27, 9.71, 6.09, 5.39, 10.86, 23.94, 20.68,
           19.39, 22.16, 32.76, 38.82, 21.33, 12.99, 5.97, 5.83, 5.07, 8.20,
           18.23, 18.93, 12.72, 16.63, 22.38, 27.76, 18.00, 12.27]
    sim = [7.70, 4.70, 4.43, 7.83, 14.55, 16.00, 15.33, 17.97, 20.37, 24.45,
           17.19, 13.27, 10.40, 6.93, 5.40, 10.19, 16.72, 22.35, 13.78, 17.44,
           24.20, 29.67, 17.56, 13.20, 12.94, 2.59, 5.60, 12.74, 21.94, 16.05,
           18.74, 17.99, 27.76, 24.48, 15.41, 12.64, 6.83, 6.11, 5.24, 11.99,
           19.88, 20.51, 12.28, 17.89, 19.38, 19.28, 14.48, 8.00]
    return pd.DataFrame({"mes": np.arange(1, 49),
                         "periodo": ["calibracion"] * 24 + ["validacion"] * 24,
                         "Q_obs_m3_s": obs, "Q_sim_m3_s": sim})


def _arreglo_fotovoltaico() -> pd.DataFrame:
    return pd.DataFrame({"configuracion": ["Base", "Optima", "Sombreado"],
                         "H_kWh_m2": [1980.0, 2035.0, 1910.0],
                         "u_rel_H": [0.04, 0.04, 0.04]})


def _entradas_vertedero() -> pd.DataFrame:
    return pd.DataFrame({"magnitud": ["C_d", "b", "h"],
                         "unidad": ["1", "m", "m"],
                         "valor": [0.620, 0.500, 0.150],
                         "u_tipica": [0.015, 0.0010, 0.0015]})


CONSTRUCTORES = {
    "cinetica_monod.csv": _cinetica_monod,
    "secado_maiz_calibracion.csv": _secado_calibracion,
    "secado_maiz_validacion.csv": _secado_validacion,
    "caudal_mensual.csv": _caudal_mensual,
    "arreglo_fotovoltaico.csv": _arreglo_fotovoltaico,
    "entradas_vertedero.csv": _entradas_vertedero,
}

CARPETA_DATOS = carpeta_datos()


def leer_datos(nombre: str) -> pd.DataFrame:
    """Lee un archivo de datos/ y lo reconstruye si no está presente."""
    ruta = CARPETA_DATOS / nombre
    if not ruta.exists():
        CONSTRUCTORES[nombre]().to_csv(ruta, index=False)
    return pd.read_csv(ruta)


print("Carpeta de datos:", CARPETA_DATOS.name)

## 1. Verificar no es validar

La Definición 4.1 del libro separa dos actividades que la
conversación cotidiana confunde. Verificar comprueba que el programa
resuelve las ecuaciones que se escribieron, y es un problema de
matemáticas y de programación. Validar comprueba que esas ecuaciones
describen el sistema real, y es un problema de ingeniería que sin
datos independientes no se puede plantear.

La Figura 4.2 del libro sitúa ambas dentro del ciclo completo, con el
lazo de revisión que se activa cuando la validación falla. El orden
es estricto, porque un error de programación se compensa con un
parámetro ajustado y el resultado parece excelente hasta que cambia
la condición de operación.

El cuaderno recorre las tres etapas en ese orden, sobre los ejemplos
resueltos del capítulo.

## 2. Verificación con solución analítica, la aleta del Ejemplo 4.1

El Ejemplo 4.1 del libro verifica un código de diferencias finitas
que resuelve la aleta rectangular de un disipador electrónico. La
Ecuación 4.2 gobierna el exceso de temperatura y admite solución
cerrada con extremo adiabático, de modo que sirve de caso de
referencia.

El extremo adiabático se trata de dos maneras, con nodo fantasma,
que conserva el segundo orden, y con una diferencia hacia atrás de
primer orden. Ambas versiones corren, ninguna levanta una excepción y
las dos producen perfiles de aspecto razonable.

In [ ]:
# Datos del Ejemplo 4.1, con unidades del Sistema Internacional.
K_ALUMINIO = 205.0     # conductividad, W/(m K)
ESPESOR = 1.0e-3       # espesor de la aleta, m
H_CONV = 80.0          # coeficiente de convección, W/(m2 K)
LARGO = 0.030          # longitud de la aleta, m
THETA_BASE = 45.0      # exceso de temperatura en la base, K

M2 = 2.0 * H_CONV / (K_ALUMINIO * ESPESOR)
M_ALETA = np.sqrt(M2)


def theta_exacta(x: np.ndarray) -> np.ndarray:
    """Solución analítica del exceso de temperatura, en K."""
    return (THETA_BASE * np.cosh(M_ALETA * (LARGO - x))
            / np.cosh(M_ALETA * LARGO))


def resolver_aleta(n_intervalos: int, borde: str):
    """Diferencias finitas con extremo adiabático de dos maneras."""
    h = LARGO / n_intervalos
    x = np.linspace(0.0, LARGO, n_intervalos + 1)
    A = np.zeros((n_intervalos + 1, n_intervalos + 1))
    b = np.zeros(n_intervalos + 1)
    A[0, 0], b[0] = 1.0, THETA_BASE
    for i in range(1, n_intervalos):
        A[i, i - 1] = 1.0 / h**2
        A[i, i] = -2.0 / h**2 - M2
        A[i, i + 1] = 1.0 / h**2
    if borde == "fantasma":
        A[-1, -2] = 2.0 / h**2
        A[-1, -1] = -2.0 / h**2 - M2
    else:
        A[-1, -2] = -1.0 / h
        A[-1, -1] = 1.0 / h
    return x, np.linalg.solve(A, b), h


print(f"m = {M_ALETA:.2f} 1/m   (el libro publica 27.94 1/m)")
print(f"mL = {M_ALETA * LARGO:.3f}   (el libro publica 0.838)")
assert abs(M_ALETA - 27.94) < 5.0e-3
assert abs(M_ALETA * LARGO - 0.838) < 5.0e-4

### 2.1 Estudio de convergencia y orden observado

El Listado 4.2 del libro devuelve dos informaciones complementarias,
las estimaciones por pares consecutivos, que revelan si el régimen
asintótico se alcanzó, y la pendiente global, que promedia el ruido.
La Definición 4.3 fija qué es el orden observado y el Teorema 4.1
asegura que las estimaciones por pares convergen al orden formal.

La celda siguiente reproduce la Tabla 4.1 del libro completa.

In [ ]:
def orden_observado(h, e):
    """Orden por pares y por regresión logarítmica, Listado 4.2."""
    h, e = np.asarray(h, float), np.asarray(e, float)
    p_par = np.log(e[:-1] / e[1:]) / np.log(h[:-1] / h[1:])
    p_reg = np.polyfit(np.log(h), np.log(e), 1)[0]
    return p_par, p_reg


mallas = [5, 10, 20, 40, 80, 160]
pasos, err_fantasma, err_atras = [], [], []
for n_int in mallas:
    x, th2, h = resolver_aleta(n_int, "fantasma")
    _, th1, _ = resolver_aleta(n_int, "atras")
    pasos.append(h)
    err_fantasma.append(np.max(np.abs(th2 - theta_exacta(x))))
    err_atras.append(np.max(np.abs(th1 - theta_exacta(x))))

p2_par, p2_reg = orden_observado(pasos, err_fantasma)
p1_par, p1_reg = orden_observado(pasos, err_atras)

tabla_4_1 = pd.DataFrame({
    "N": mallas,
    "h_mm": np.array(pasos) * 1e3,
    "e_inf_fantasma_K": err_fantasma,
    "p_fantasma": [np.nan, *p2_par],
    "e_inf_atras_K": err_atras,
    "p_atras": [np.nan, *p1_par]})
print(tabla_4_1.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print(f"\nRegresión global, nodo fantasma p = {p2_reg:.3f}")
print(f"Regresión global, diferencia hacia atrás p = {p1_reg:.3f}")
print(f"Razón de errores con N = 160: {err_atras[-1] / err_fantasma[-1]:.0f}")

### 2.2 Comparación con la Tabla 4.1 del libro

La verificación de este cuaderno consiste en comprobar que cada celda
de la tabla reproduce la cifra publicada. Los errores del libro
aparecen con cuatro cifras significativas y los órdenes con tres.

In [ ]:
# Cifras publicadas en la Tabla 4.1 del libro.
LIBRO_TABLA_4_1 = pd.DataFrame({
    "N": [5, 10, 20, 40, 80, 160],
    "e_inf_fantasma_K": [2.196e-2, 5.504e-3, 1.377e-3,
                         3.443e-4, 8.607e-5, 2.152e-5],
    "p_fantasma": [np.nan, 1.997, 1.999, 2.000, 2.000, 2.000],
    "e_inf_atras_K": [2.011e0, 9.735e-1, 4.787e-1,
                      2.373e-1, 1.181e-1, 5.894e-2],
    "p_atras": [np.nan, 1.047, 1.024, 1.012, 1.006, 1.003]})

for columna in ("e_inf_fantasma_K", "e_inf_atras_K"):
    obtenido = tabla_4_1[columna].to_numpy()
    publicado = LIBRO_TABLA_4_1[columna].to_numpy()
    assert np.allclose(obtenido, publicado, rtol=5.0e-4), columna
for columna in ("p_fantasma", "p_atras"):
    obtenido = tabla_4_1[columna].to_numpy()[1:]
    publicado = LIBRO_TABLA_4_1[columna].to_numpy()[1:]
    assert np.allclose(obtenido, publicado, atol=5.0e-4), columna

print("La Tabla 4.1 del libro se reproduce celda a celda.")
print(f"El libro anuncia una razón de errores de 2740 y aquí sale "
      f"{err_atras[-1] / err_fantasma[-1]:.0f}.")

### 2.3 Lectura en escala logarítmica

La Figura 4.4 del libro presenta este mismo estudio con el error en
escala logarítmica, donde la relación de potencia se convierte en una
recta cuya pendiente es el orden, y con la sucesión de órdenes
observados al lado. La celda siguiente la reconstruye.

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(10.5, 4.2))

h_mm = np.array(pasos) * 1e3
izq.loglog(h_mm, err_fantasma, "o-", color=PALETA["azul"],
           label="frontera de segundo orden")
izq.loglog(h_mm, err_atras, "s-", color=PALETA["rojo"],
           label="frontera de primer orden")
referencia = np.array([h_mm.min(), h_mm.max()])
izq.loglog(referencia, 6.0e-4 * referencia**2, "--",
           color=PALETA["gris"], lw=0.9)
izq.loglog(referencia, 3.0e-1 * referencia, ":",
           color=PALETA["gris"], lw=0.9)
izq.set_xlabel("Paso de malla h (mm)")
izq.set_ylabel("Error en norma del máximo (K)")
izq.set_title("(a) error frente al refinamiento")
izq.legend(loc="lower right", fontsize=8)

h_medio = np.sqrt(h_mm[:-1] * h_mm[1:])
der.semilogx(h_medio, p2_par, "o-", color=PALETA["azul"])
der.semilogx(h_medio, p1_par, "s-", color=PALETA["rojo"])
der.axhline(2.0, color=PALETA["gris"], ls="--", lw=0.9)
der.axhline(1.0, color=PALETA["gris"], ls=":", lw=0.9)
der.set_ylim(0.6, 2.4)
der.set_xlabel("Media geométrica de h (mm)")
der.set_ylabel("Orden observado")
der.set_title("(b) orden observado por pares de mallas")

fig.tight_layout()
plt.show()

## 3. Extrapolación de Richardson e índice de convergencia de malla

Establecido el régimen asintótico, la Ecuación 4.3 del libro elimina
el término dominante del error con dos soluciones de pasos distintos,
y la Ecuación 4.4 convierte esa diferencia en el índice de
convergencia de malla, con el factor de seguridad de 1.25 que
corresponde al caso de tres mallas con orden observado.

El libro toma como magnitud integral el calor disipado por unidad de
ancho, de valor exacto 176.4898 W/m, y reporta 176.6111 W/m,
176.5201 W/m y 176.4974 W/m para las mallas de 10, 20 y 40
intervalos.

In [ ]:
def calor_disipado(n_intervalos: int) -> float:
    """Calor por unidad de ancho, integrando la convección, en W/m."""
    x, theta, _ = resolver_aleta(n_intervalos, "fantasma")
    return float(np.trapezoid(2.0 * H_CONV * theta, x))


q_exacto = (np.sqrt(2.0 * H_CONV * K_ALUMINIO * ESPESOR) * THETA_BASE
            * np.tanh(M_ALETA * LARGO))
q_mallas = [calor_disipado(n) for n in (10, 20, 40)]

print(f"Calor exacto        {q_exacto:.4f} W/m   (libro 176.4898 W/m)")
for n, q in zip((10, 20, 40), q_mallas):
    print(f"Malla de {n:2d} intervalos {q:.4f} W/m")
assert abs(q_exacto - 176.4898) < 5.0e-4
assert np.allclose(q_mallas, [176.6111, 176.5201, 176.4974], atol=5.0e-5)

### 3.1 El Listado 4.3 del libro sobre las cifras publicadas

El Listado 4.3 aplica la extrapolación a los tres valores del calor
disipado tal como aparecen impresos, con cuatro decimales. Conviene
reproducirlo con esos mismos literales, porque el estimador de tres
mallas es un cociente de diferencias y amplifica el redondeo de la
entrada.

In [ ]:
def richardson(f_1: float, f_2: float, f_3: float,
               r: float = 2.0, fs: float = 1.25):
    """f_1 en la malla gruesa y f_3 en la fina, razón r, Listado 4.3."""
    p = np.log(abs((f_1 - f_2) / (f_2 - f_3))) / np.log(r)
    f_ext = f_3 + (f_3 - f_2) / (r**p - 1.0)
    gci = fs * abs((f_3 - f_2) / f_3) / (r**p - 1.0)
    return p, f_ext, gci


p_libro, q_ext, gci = richardson(176.6111, 176.5201, 176.4974)
error_verdadero = abs(176.4974 - q_exacto) / q_exacto

print(f"Orden de tres mallas   {p_libro:.3f}      (libro 2.003)")
print(f"Valor extrapolado      {q_ext:.4f} W/m (libro 176.4899 W/m)")
print(f"Índice de convergencia {100 * gci:.4f} %  (libro 0.0053 %)")
print(f"Error verdadero        {100 * error_verdadero:.4f} %  "
      f"(libro 0.0043 %)")
assert abs(p_libro - 2.003) < 5.0e-4
assert abs(q_ext - 176.4899) < 5.0e-5
assert abs(100 * gci - 0.0053) < 5.0e-5
assert abs(100 * error_verdadero - 0.0043) < 5.0e-5

### 3.2 Qué cambia con las cifras sin redondear

El mismo estimador alimentado con los valores en doble precisión
devuelve un orden algo distinto. La diferencia no es un error del
libro ni del cuaderno, es la sensibilidad del cociente de diferencias
al redondeo, y conviene verla una vez para no confundirla más
adelante con un defecto del código.

In [ ]:
p_pleno, q_ext_pleno, gci_pleno = richardson(*q_mallas)
print(f"Con cuatro decimales publicados  p = {p_libro:.4f}")
print(f"Con la precisión de la máquina   p = {p_pleno:.4f}")
print(f"El valor extrapolado apenas se mueve, "
      f"{q_ext:.5f} frente a {q_ext_pleno:.5f} W/m")
print(f"y ambos caen sobre el exacto {q_exacto:.5f} W/m.")
assert abs(q_ext_pleno - q_exacto) < 1.0e-4

## 4. Solución fabricada, la columna de suelo del Ejemplo 4.2

La Definición 4.2 del libro invierte el planteamiento y escoge
primero la solución, para construir después la ecuación que la tiene
por respuesta exacta. La Figura 4.3 resume los cinco pasos.

El Ejemplo 4.2 verifica un modelo de transporte reactivo en una
columna de suelo de 0.40 m, con coeficiente de difusión que crece
linealmente con la profundidad. La Ecuación 4.5 no admite solución
cerrada elemental, de modo que ninguna solución analítica cubre el
operador completo.

El Listado 4.1 del libro genera el término fuente con álgebra
simbólica, que elimina el riesgo de derivar a mano una expresión
larga y entrega además la función evaluable.

In [ ]:
import sympy as sp

x, L, D0, k, c0 = sp.symbols("x L D_0 k c_0", positive=True)
D = D0 * (1 + x / L)
c_fab = c0 * sp.cos(sp.pi * x / (2 * L))
flujo = D * sp.diff(c_fab, x)
fuente = sp.simplify(-sp.diff(flujo, x) + k * c_fab)
f_num = sp.lambdify((x, L, D0, k, c0), fuente, "numpy")

valores = (0.40, 5.0e-10, 1.0e-6, 50.0)   # L, D0, k, c0
f_centro = f_num(0.20, *valores)

print("Término fuente construido con SymPy:")
sp.pprint(sp.simplify(sp.expand(fuente)))
print(f"\nf en el centro de la columna: {f_centro:.4e} mg/(L s)")
print("El libro publica 3.594e-05 mg/(L s).")
assert abs(f_centro - 3.594e-5) / 3.594e-5 < 5.0e-4

### 4.1 La forma cerrada de la Ecuación 4.6

El libro publica el término fuente ya desarrollado. Comprobar que la
expresión simbólica y la fórmula impresa coinciden en todo el dominio
es una verificación barata que atrapa un error de transcripción.

In [ ]:
L_col, D_0, k_reac, c_ref = valores


def fuente_libro(x_m: np.ndarray) -> np.ndarray:
    """Ecuación 4.6 del libro, evaluada punto a punto."""
    arg = np.pi * x_m / (2.0 * L_col)
    return (np.pi * c_ref * D_0 / (2.0 * L_col**2)
            * (np.sin(arg) + np.pi / 2.0 * (1.0 + x_m / L_col) * np.cos(arg))
            + k_reac * c_ref * np.cos(arg))


malla_prueba = np.linspace(0.0, L_col, 41)
assert np.allclose(f_num(malla_prueba, *valores), fuente_libro(malla_prueba))
print("La expresión simbólica y la Ecuación 4.6 coinciden en 41 puntos.")

### 4.2 Orden observado del esquema con coeficiente variable

El coeficiente de difusión se promedia en las caras de cada celda,
que es justamente la ruta del código donde se esconden los defectos.
El libro reporta un error que pasa de 2.382e-3 mg/L a 2.343e-6 mg/L y
órdenes de 1.991, 1.999, 1.999, 2.000 y 2.000.

In [ ]:
def resolver_columna(n_intervalos: int):
    """Volúmenes finitos con el coeficiente promediado en las caras."""
    h = L_col / n_intervalos
    nodos = np.linspace(0.0, L_col, n_intervalos + 1)
    caras = 0.5 * (nodos[:-1] + nodos[1:])
    d_cara = D_0 * (1.0 + caras / L_col)
    A = np.zeros((n_intervalos + 1, n_intervalos + 1))
    b = np.zeros(n_intervalos + 1)
    A[0, 0], b[0] = 1.0, c_ref
    A[-1, -1], b[-1] = 1.0, 0.0
    for i in range(1, n_intervalos):
        A[i, i - 1] = -d_cara[i - 1] / h**2
        A[i, i] = (d_cara[i - 1] + d_cara[i]) / h**2 + k_reac
        A[i, i + 1] = -d_cara[i] / h**2
        b[i] = f_num(nodos[i], *valores)
    return nodos, np.linalg.solve(A, b)


mallas_columna = [8, 16, 32, 64, 128, 256]
errores_columna = []
for n_int in mallas_columna:
    nodos, c_num = resolver_columna(n_int)
    c_ref_nodos = c_ref * np.cos(np.pi * nodos / (2.0 * L_col))
    errores_columna.append(np.max(np.abs(c_num - c_ref_nodos)))

ordenes_columna, _ = orden_observado(
    L_col / np.array(mallas_columna), errores_columna)

print("N      error (mg/L)   orden observado")
for i, n_int in enumerate(mallas_columna):
    orden = f"{ordenes_columna[i - 1]:.3f}" if i else "---"
    print(f"{n_int:3d}    {errores_columna[i]:.4e}   {orden:>8}")

assert abs(errores_columna[0] - 2.382e-3) / 2.382e-3 < 5.0e-4
assert abs(errores_columna[-1] - 2.343e-6) / 2.343e-6 < 5.0e-4
assert np.allclose(ordenes_columna,
                   [1.991, 1.999, 1.999, 2.000, 2.000], atol=5.0e-4)
print("\nLos cinco órdenes coinciden con los que publica el Ejemplo 4.2.")

## 5. Calibración, la cinética de Monod del Ejemplo 4.3

Verificado el código, el modelo sigue sin poder predecir nada porque
sus parámetros son desconocidos. La Definición 4.4 del libro trata la
calibración como un problema de optimización sobre la función
objetivo de la Ecuación 4.7, y el Teorema 4.2 permite acompañar cada
parámetro de una medida de su precisión.

El Ejemplo 4.3 mide la velocidad específica de crecimiento de una
levadura sobre melaza diluida, a diez concentraciones de sustrato
entre 0.5 g/L y 35 g/L, con repetibilidad de 0.012 1/h.

In [ ]:
from scipy.optimize import curve_fit, least_squares
from scipy.stats import t as t_student

monod_datos = leer_datos("cinetica_monod.csv")
S_sustrato = monod_datos["S_g_L"].to_numpy()
mu_medida = monod_datos["mu_1_h"].to_numpy()


def monod(S, mu_max, K_s):
    """Ecuación 4.8 del libro, velocidad específica en 1/h."""
    return mu_max * S / (K_s + S)


theta, cov = curve_fit(monod, S_sustrato, mu_medida, p0=[0.3, 1.0])
n_datos, m_par = S_sustrato.size, theta.size
ee = np.sqrt(np.diag(cov))
t_critico = t_student.ppf(0.975, n_datos - m_par)
intervalos = np.column_stack([theta - t_critico * ee,
                              theta + t_critico * ee])
correlacion = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])

residuales = mu_medida - monod(S_sustrato, *theta)
suma_cuadrados = float(residuales @ residuales)
s_residual = np.sqrt(suma_cuadrados / (n_datos - m_par))
r2 = 1.0 - suma_cuadrados / np.sum((mu_medida - mu_medida.mean())**2)

print(f"mu_max = {theta[0]:.4f} +- {ee[0]:.4f} 1/h    "
      f"(libro 0.4137 +- 0.0074)")
print(f"K_s    = {theta[1]:.4f} +- {ee[1]:.4f} g/L    "
      f"(libro 2.231 +- 0.163)")
print(f"t de Student con {n_datos - m_par} grados de libertad: "
      f"{t_critico:.3f}   (libro 2.306)")
print(f"Intervalo de mu_max: {intervalos[0, 0]:.4f} a {intervalos[0, 1]:.4f}"
      f"   (libro 0.3966 a 0.4308)")
print(f"Intervalo de K_s:    {intervalos[1, 0]:.3f} a {intervalos[1, 1]:.3f}"
      f"       (libro 1.855 a 2.606)")
print(f"Desviación residual {s_residual:.4f} 1/h   (libro 0.0104)")
print(f"Coeficiente de determinación {r2:.4f}   (libro 0.9923)")
print(f"Correlación entre parámetros {correlacion:.3f}   (libro 0.778)")

assert abs(theta[0] - 0.4137) < 5.0e-5 and abs(ee[0] - 0.0074) < 5.0e-5
assert abs(theta[1] - 2.231) < 5.0e-4 and abs(ee[1] - 0.163) < 5.0e-4
assert abs(t_critico - 2.306) < 5.0e-4
assert abs(s_residual - 0.0104) < 5.0e-5
assert abs(r2 - 0.9923) < 5.0e-5
assert abs(correlacion - 0.778) < 5.0e-4

### 5.1 Residuales ponderados y límites físicos

Cuando el problema exige límites físicos, pesos distintos o una
pérdida robusta, la rutina de mínimos cuadrados general del Listado
4.5 resulta preferible. El mismo óptimo aparece por los dos caminos,
lo cual es una comprobación de consistencia que cuesta tres líneas.

In [ ]:
sigma_instrumento = np.full(S_sustrato.size, 0.012)   # 1/h


def residuo_ponderado(parametros):
    """Residuales divididos por la desviación del instrumento."""
    return (monod(S_sustrato, *parametros) - mu_medida) / sigma_instrumento


limites = ([0.0, 0.0], [2.0, 50.0])
solucion = least_squares(residuo_ponderado, x0=[0.3, 1.0], bounds=limites)
cov_ponderada = np.linalg.inv(solucion.jac.T @ solucion.jac)
ee_ponderado = np.sqrt(np.diag(cov_ponderada))
chi2_reducido = 2 * solucion.cost / (S_sustrato.size - solucion.x.size)

print(f"Óptimo con límites   {solucion.x[0]:.5f} 1/h y "
      f"{solucion.x[1]:.5f} g/L")
print(f"Óptimo sin límites   {theta[0]:.5f} 1/h y {theta[1]:.5f} g/L")
print(f"Chi cuadrado reducido {chi2_reducido:.3f}, menor que uno porque "
      f"la dispersión real es menor que la repetibilidad declarada.")
assert np.allclose(solucion.x, theta, rtol=1.0e-5)

### 5.2 Bootstrap de los residuales

El Teorema 4.2 descansa sobre la linealidad local, que falla cuando
la función objetivo tiene un valle curvo y alargado. El remuestreo por
bootstrap ofrece intervalos que no dependen de esa linealización. El
libro reporta, con 4000 réplicas y semilla 20262, intervalos de 0.4020
a 0.4277 y de 1.956 a 2.524.

In [ ]:
def bootstrap_residuales(n_replicas: int = 4000, semilla: int = SEMILLA):
    """Remuestreo de los residuales del ajuste, con semilla declarada."""
    generador = np.random.default_rng(semilla)
    ajustado = monod(S_sustrato, *theta)
    muestras = np.empty((n_replicas, 2))
    for i in range(n_replicas):
        y = ajustado + generador.choice(residuales, S_sustrato.size,
                                        replace=True)
        muestras[i], _ = curve_fit(monod, S_sustrato, y, p0=theta,
                                   maxfev=20000)
    return muestras


replicas = bootstrap_residuales()
ic_mu = np.percentile(replicas[:, 0], [2.5, 97.5])
ic_ks = np.percentile(replicas[:, 1], [2.5, 97.5])

print(f"Bootstrap mu_max {ic_mu[0]:.4f} a {ic_mu[1]:.4f}   "
      f"(libro 0.4020 a 0.4277)")
print(f"Bootstrap K_s    {ic_ks[0]:.3f} a {ic_ks[1]:.3f}       "
      f"(libro 1.956 a 2.524)")
print("Ambos son algo más estrechos que los de la aproximación lineal,")
print("de modo que la linealización es aceptable en este caso.")
assert np.allclose(ic_mu, [0.4020, 0.4277], atol=5.0e-5)
assert np.allclose(ic_ks, [1.956, 2.524], atol=5.0e-4)

### 5.3 La geometría que explica la correlación

La Figura 4.5 del libro muestra las curvas de nivel de la suma de
cuadrados con la región de confianza conjunta y la nube del
bootstrap, y al lado el ajuste con su banda de confianza. Una
correlación de 0.778 significa que un valor mayor de la velocidad
máxima se compensa con una semisaturación mayor, y por eso la región
conjunta es una elipse inclinada.

In [ ]:
from scipy import stats

fig, (izq, der) = plt.subplots(1, 2, figsize=(10.5, 4.4))

mu_malla = np.linspace(0.385, 0.445, 160)
ks_malla = np.linspace(1.45, 3.05, 160)
MU, KS = np.meshgrid(mu_malla, ks_malla)
objetivo = np.array([[np.sum((mu_medida - monod(S_sustrato, a, b))**2)
                      for a, b in zip(fila_a, fila_b)]
                     for fila_a, fila_b in zip(MU, KS)])

niveles = np.array([1.2, 1.6, 2.4, 4.0, 7.0, 12.0]) * suma_cuadrados
izq.contour(MU, KS, objetivo, levels=niveles, colors=PALETA["gris"],
            linewidths=0.7)
izq.plot(replicas[:, 0], replicas[:, 1], ".", ms=1.0,
         color=PALETA["verde"], alpha=0.30, label="remuestreo bootstrap")
angulo = np.linspace(0.0, 2 * np.pi, 400)
radio2 = m_par * stats.f.ppf(0.95, m_par, n_datos - m_par)
elipse = theta[:, None] + np.sqrt(radio2) * (
    np.linalg.cholesky(cov) @ np.vstack([np.cos(angulo), np.sin(angulo)]))
izq.plot(elipse[0], elipse[1], "-", color=PALETA["rojo"], lw=1.3,
         label="región conjunta al 95 por ciento")
izq.plot(theta[0], theta[1], "o", color=PALETA["azul"], ms=5,
         label="óptimo")
izq.set_xlabel("mu_max (1/h)")
izq.set_ylabel("K_s (g/L)")
izq.set_title("(a) función objetivo")
izq.legend(loc="lower right", fontsize=7)

s_fino = np.linspace(0.0, 38.0, 400)
gradiente = np.vstack([s_fino / (theta[1] + s_fino),
                       -theta[0] * s_fino / (theta[1] + s_fino)**2]).T
var_ajuste = np.einsum("ij,jk,ik->i", gradiente, cov, gradiente)
ajuste = monod(s_fino, *theta)
der.fill_between(s_fino, ajuste - t_critico * np.sqrt(var_ajuste),
                 ajuste + t_critico * np.sqrt(var_ajuste),
                 color=PALETA["rojo"], alpha=0.16, lw=0,
                 label="banda al 95 por ciento")
der.plot(s_fino, ajuste, "-", color=PALETA["rojo"], label="modelo ajustado")
der.plot(S_sustrato, mu_medida, "o", color=PALETA["azul"],
         label="mediciones")
der.set_xlabel("Sustrato S (g/L)")
der.set_ylabel("mu (1/h)")
der.set_ylim(0.0, 0.46)
der.set_title("(b) ajuste y banda de confianza")
der.legend(loc="lower right", fontsize=7)

fig.tight_layout()
plt.show()

## 6. Identificabilidad y parsimonia, el secado del Ejemplo 4.4

La Definición 4.6 del libro distingue la identificabilidad
estructural de la práctica. La primera se detecta por análisis del
propio modelo y es irreparable con más datos; la segunda depende del
diseño experimental y del ruido, y se diagnostica sobre la matriz de
sensibilidad escalada con el número de condición y con el índice de
colinealidad de la Ecuación 4.9.

El Ejemplo 4.4 registra la razón de humedad de maíz a 60 grados
Celsius en trece instantes entre 0.25 h y 8 h, y compara cuatro
modelos de la literatura de poscosecha.

In [ ]:
secado = leer_datos("secado_maiz_calibracion.csv")
t_secado = secado["t_h"].to_numpy()
mr_secado = secado["MR"].to_numpy()


def exponencial(t, k):
    """Modelo exponencial simple, un parámetro."""
    return np.exp(-k * t)


def henderson_pabis(t, a, k):
    """Modelo de Henderson y Pabis, dos parámetros."""
    return a * np.exp(-k * t)


def page(t, k, n):
    """Modelo de Page, dos parámetros."""
    return np.exp(-k * t**n)


def dos_terminos(t, a, k_0, k_1):
    """Modelo de dos términos, tres parámetros."""
    return a * np.exp(-k_0 * t) + (1.0 - a) * np.exp(-k_1 * t)


print(secado.to_string(index=False))
print(f"\nSon {len(secado)} instantes entre {t_secado.min()} h "
      f"y {t_secado.max()} h.")

### 6.1 Sensibilidad escalada, número de condición y colinealidad

El Listado 4.6 del libro escala cada columna de la matriz de
sensibilidad por el valor del parámetro, de modo que las columnas
queden comparables aunque los parámetros tengan unidades distintas.
De manera orientativa, un índice de colinealidad por debajo de 5
corresponde a parámetros bien separables y por encima de 20 señala
combinaciones que los datos no distinguen.

In [ ]:
def sensibilidad_escalada(modelo, parametros, x, rel=1.0e-6):
    """Columna j igual a theta_j por la derivada, Listado 4.6."""
    J = np.zeros((np.size(x), np.size(parametros)))
    for j in range(np.size(parametros)):
        d = rel * max(abs(parametros[j]), 1.0e-12)
        mas = np.array(parametros, float)
        menos = np.array(parametros, float)
        mas[j], menos[j] = mas[j] + d, menos[j] - d
        dif = modelo(x, *mas) - modelo(x, *menos)
        J[:, j] = dif / (2 * d) * parametros[j]
    return J


def colinealidad(J):
    """Índice de colinealidad de la Ecuación 4.9."""
    Jn = J / np.linalg.norm(J, axis=0)
    return 1.0 / np.sqrt(np.linalg.eigvalsh(Jn.T @ Jn).min())


def aic_corregido(suma, n, k_par):
    """Ecuación 4.10, con k_par igual al número de parámetros más la escala."""
    return n * np.log(suma / n) + 2 * k_par + 2 * k_par * (k_par + 1) / (n - k_par - 1)

### 6.2 La Tabla 4.2 del libro reproducida

La última columna recoge el error de predicción fuera de la muestra
con validación cruzada de cinco bloques y semilla 20262. La
coincidencia entre el ordenamiento del criterio de información y el
de la validación cruzada refuerza la conclusión.

El modelo de dos términos se ajusta desde un punto inicial que lo
lleva al punto donde las dos constantes coinciden, que es el que el
libro reporta y el que delata la falta de identificabilidad práctica.

In [ ]:
CANDIDATOS = {
    "Exponencial simple": (exponencial, [0.3]),
    "Henderson y Pabis": (henderson_pabis, [1.0, 0.3]),
    "Page": (page, [0.3, 1.0]),
    "Dos términos": (dos_terminos, [0.2, 0.9, 0.1]),
}


def rmse_validacion_cruzada(modelo, p0, n_bloques=5, semilla=SEMILLA):
    """Error fuera de la muestra con bloques barajados y semilla fija."""
    generador = np.random.default_rng(semilla)
    orden = generador.permutation(t_secado.size)
    cuadrados = []
    for bloque in np.array_split(orden, n_bloques):
        entrena = np.setdiff1d(orden, bloque)
        ajuste, _ = curve_fit(modelo, t_secado[entrena], mr_secado[entrena],
                              p0=p0, maxfev=200000)
        cuadrados.extend((mr_secado[bloque]
                          - modelo(t_secado[bloque], *ajuste))**2)
    return float(np.sqrt(np.mean(cuadrados)))


filas = []
ajustes_secado = {}
for nombre, (modelo, p0) in CANDIDATOS.items():
    par, _ = curve_fit(modelo, t_secado, mr_secado, p0=p0, maxfev=200000)
    suma = float(np.sum((mr_secado - modelo(t_secado, *par))**2))
    m_mod = len(par)
    J = sensibilidad_escalada(modelo, par, t_secado)
    ajustes_secado[nombre] = par
    filas.append({
        "Modelo": nombre, "m": m_mod, "S": suma,
        "s": np.sqrt(suma / (t_secado.size - m_mod)),
        "AICc": aic_corregido(suma, t_secado.size, m_mod + 1),
        "RMSE_vc": rmse_validacion_cruzada(modelo, p0),
        "kappa": np.linalg.cond(J),
        "gamma": colinealidad(J) if m_mod > 1 else 1.0})

tabla_4_2 = pd.DataFrame(filas)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print(tabla_4_2.to_string(index=False))
pd.reset_option("display.float_format")

In [ ]:
# Cifras publicadas en la Tabla 4.2 del libro.
LIBRO_TABLA_4_2 = pd.DataFrame({
    "Modelo": ["Exponencial simple", "Henderson y Pabis", "Page",
               "Dos términos"],
    "S": [9.971e-3, 3.173e-3, 5.815e-4, 9.971e-3],
    "s": [0.0288, 0.0170, 0.0073, 0.0316],
    "AICc": [-88.05, -99.47, -121.53, -80.25],
    "RMSE_vc": [0.0292, 0.0196, 0.0075, 0.0292]})

comparacion = tabla_4_2.merge(LIBRO_TABLA_4_2, on="Modelo",
                              suffixes=("_cuaderno", "_libro"))
for columna in ("S", "s", "AICc", "RMSE_vc"):
    a = comparacion[f"{columna}_cuaderno"].to_numpy()
    b = comparacion[f"{columna}_libro"].to_numpy()
    tolerancia = 5.0e-3 if columna == "AICc" else 1.0e-3
    assert np.allclose(a, b, rtol=tolerancia, atol=5.0e-5), columna

page_par = ajustes_secado["Page"]
print(f"Page: k = {page_par[0]:.4f} y n = {page_par[1]:.4f}   "
      f"(libro 0.3430 y 1.1627)")
fila_page = tabla_4_2.set_index("Modelo").loc["Page"]
print(f"Page: número de condición {fila_page['kappa']:.2f} "
      f"(libro 3.18) e índice de colinealidad "
      f"{fila_page['gamma']:.2f} (libro 2.31)")
fila_dos = tabla_4_2.set_index("Modelo").loc["Dos términos"]
print(f"Dos términos: número de condición {fila_dos['kappa']:.1e} "
      f"(el libro publica 1.3e10, del mismo orden de magnitud)")
assert abs(page_par[0] - 0.3430) < 5.0e-5
assert abs(page_par[1] - 1.1627) < 1.0e-4
assert abs(fila_page["kappa"] - 3.18) < 5.0e-3
assert abs(fila_page["gamma"] - 2.31) < 5.0e-3
assert fila_dos["kappa"] > 1.0e8
print("\nLa Tabla 4.2 se reproduce en sus cuatro columnas numéricas.")

### 6.3 Equifinalidad, dos puntos iniciales y dos respuestas

El libro advierte que dos puntos iniciales distintos entregan
conjuntos de parámetros distintos con sumas de cuadrados
prácticamente iguales. Ese es el fenómeno de la equifinalidad, y el
optimizador no lo denuncia por sí solo.

In [ ]:
for inicial in ([0.2, 0.9, 0.1], [0.5, 0.3, 0.05], [0.8, 0.1, 0.6]):
    par, _ = curve_fit(dos_terminos, t_secado, mr_secado, p0=inicial,
                       maxfev=200000)
    suma = np.sum((mr_secado - dos_terminos(t_secado, *par))**2)
    print(f"desde {str(inicial):>18} -> a = {par[0]:9.4f}, "
          f"k_0 = {par[1]:.5f}, k_1 = {par[2]:.5f}, S = {suma:.4e}")
print("\nTres puntos iniciales, tres conjuntos de parámetros, y un ajuste")
print("que en el mejor caso empata con el del modelo de un solo parámetro.")

### 6.4 Falta de identificabilidad estructural

El libro cierra el ejemplo con el modelo de difusión en lámina plana
truncado al primer término, cuyos dos parámetros aparecen solo en el
cociente entre la difusividad efectiva y el cuadrado del semiespesor.
La pareja de 3.07e-10 m2/s con 3.0 mm y la de 1.36e-10 m2/s con 2.0 mm
producen predicciones idénticas, de modo que ningún dato podrá
separarlas.

In [ ]:
def difusion_truncada(t_h, d_efectiva, semiespesor):
    """Primer término de la serie, difusividad en m2/s y t en horas."""
    cociente = d_efectiva * 3600.0 / semiespesor**2      # 1/h
    return (8.0 / np.pi**2) * np.exp(-(np.pi**2 / 4.0) * cociente * t_h)


pareja_a = difusion_truncada(t_secado, 3.07e-10, 3.0e-3)
pareja_b = difusion_truncada(t_secado, 1.36e-10, 2.0e-3)
print(f"Cociente de la pareja A: {3.07e-10 * 3600 / 3.0e-3**2:.4f} 1/h")
print(f"Cociente de la pareja B: {1.36e-10 * 3600 / 2.0e-3**2:.4f} 1/h")
print(f"Diferencia máxima entre las dos predicciones: "
      f"{np.max(np.abs(pareja_a - pareja_b)):.2e}")
assert np.max(np.abs(pareja_a - pareja_b)) < 1.0e-3
print("Ningún dato podrá separar las dos parejas, porque el modelo solo")
print("depende del cociente entre difusividad y semiespesor al cuadrado.")

La reparametrización con ese cociente único convierte un modelo de
dos parámetros no identificables en uno de un solo parámetro
perfectamente determinado. El libro reporta 0.1226 con incertidumbre
de 0.0149 en 1/h y número de condición unitario.

In [ ]:
def difusion_reparametrizada(t_h, cociente):
    """Una sola combinación identificable, el cociente en 1/h."""
    return (8.0 / np.pi**2) * np.exp(-(np.pi**2 / 4.0) * cociente * t_h)


q_par, q_cov = curve_fit(difusion_reparametrizada, t_secado, mr_secado,
                         p0=[0.1])
q_ee = float(np.sqrt(q_cov[0, 0]))
J_q = sensibilidad_escalada(difusion_reparametrizada, q_par, t_secado)
print(f"Cociente reparametrizado {q_par[0]:.4f} +- {q_ee:.4f} 1/h   "
      f"(libro 0.1226 +- 0.0149)")
print(f"Número de condición {np.linalg.cond(J_q):.2f}   (libro 1.00)")
assert abs(q_par[0] - 0.1226) < 5.0e-5
assert abs(q_ee - 0.0149) < 5.0e-5
assert abs(np.linalg.cond(J_q) - 1.0) < 1.0e-6

## 7. Validación con datos independientes, el caudal del Ejemplo 4.5

La Definición 4.7 del libro exige que el conjunto de validación no
intervenga en ninguna etapa de la estimación, ni siquiera en la
selección de la estructura o en el descarte de atípicos. En una serie
temporal la separación cronológica es obligatoria, porque partir al
azar deja puntos vecinos en ambos bloques y la correlación temporal
simula una capacidad predictiva inexistente.

El Ejemplo 4.5 calibra un modelo lluvia y escorrentía de una cuenca
del Caribe colombiano con veinticuatro meses y lo valida con los
veinticuatro siguientes, que incluyen un periodo más húmedo. El
criterio de aceptación se declaró antes del ejercicio, según pide el
Algoritmo 4.3, y exige coeficiente de eficiencia superior a 0.65 y
sesgo porcentual inferior al 15 por ciento en valor absoluto.

In [ ]:
caudal = leer_datos("caudal_mensual.csv")
calibracion = caudal[caudal["periodo"] == "calibracion"]
validacion = caudal[caudal["periodo"] == "validacion"]

CRITERIO_ACEPTACION = {"NSE_minimo": 0.65, "PBIAS_maximo": 15.0}
print("Criterio declarado antes de mirar los datos de validación:")
print(CRITERIO_ACEPTACION)
print(f"\nMeses de calibración {len(calibracion)}, "
      f"de validación {len(validacion)}")
print(f"Caudal medio observado en calibración "
      f"{calibracion['Q_obs_m3_s'].mean():.2f} m3/s")
print(f"Caudal medio observado en validación "
      f"{validacion['Q_obs_m3_s'].mean():.2f} m3/s")

### 7.1 Las siete métricas del Listado 4.7

El libro insiste en devolverlas todas a la vez, para que el informe
nunca reporte una sin las demás. La Tabla 4.3 resume qué mide cada
una y en qué situación engaña.

In [ ]:
def metricas(obs, sim) -> dict[str, float]:
    """Métricas de uso corriente en validación, Listado 4.7."""
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    e = sim - obs
    rmse = np.sqrt(np.mean(e**2))
    dif = np.abs(sim - obs.mean()) + np.abs(obs - obs.mean())
    den_d = np.sum(dif**2)
    return {"ME": e.mean(), "MAE": np.abs(e).mean(), "RMSE": rmse,
            "PBIAS": 100 * e.sum() / obs.sum(),
            "NSE": 1 - np.sum(e**2) / np.sum((obs - obs.mean())**2),
            "RSR": rmse / obs.std(), "d": 1 - np.sum(e**2) / den_d}


m_cal = metricas(calibracion["Q_obs_m3_s"], calibracion["Q_sim_m3_s"])
m_val = metricas(validacion["Q_obs_m3_s"], validacion["Q_sim_m3_s"])
tabla_metricas = pd.DataFrame({"calibracion": m_cal, "validacion": m_val})
print(tabla_metricas.to_string(float_format=lambda v: f"{v:8.4f}"))

# Cifras del Ejemplo 4.5, con el número de decimales con que se
# publican. La comparación admite una unidad en la última cifra
# impresa, que es la tolerancia que el propio redondeo impone.
PUBLICADO_4_5 = [
    ("calibración", "NSE", m_cal["NSE"], 0.951, 3),
    ("calibración", "RSR", m_cal["RSR"], 0.222, 3),
    ("calibración", "PBIAS", m_cal["PBIAS"], 0.04, 2),
    ("calibración", "RMSE", m_cal["RMSE"], 1.58, 2),
    ("calibración", "ME", m_cal["ME"], 0.01, 2),
    ("validación", "NSE", m_val["NSE"], 0.747, 3),
    ("validación", "RSR", m_val["RSR"], 0.504, 3),
    ("validación", "PBIAS", m_val["PBIAS"], -11.5, 1),
    ("validación", "RMSE", m_val["RMSE"], 4.42, 2),
    ("validación", "ME", m_val["ME"], -1.89, 2),
]

print("periodo       métrica  cuaderno    libro   diferencia")
for periodo, nombre, obtenido, libro_valor, decimales in PUBLICADO_4_5:
    diferencia = abs(obtenido - libro_valor)
    print(f"{periodo:12s}  {nombre:6s}  {obtenido:9.4f} "
          f"{libro_valor:8.3f}   {diferencia:.1e}")
    assert diferencia <= 10.0**(-decimales), (periodo, nombre)
print("\nLas diez cifras del Ejemplo 4.5 se reproducen dentro de una")
print("unidad de la última cifra que el libro imprime.")

Una cifra merece comentario. El coeficiente de eficiencia de la
validación vale 0.74647, que redondeado a tres decimales da 0.746,
mientras que el libro imprime 0.747. La diferencia proviene de un
doble redondeo, pues 0.74647 se convierte primero en 0.7465 y de ahí
en 0.747. El valor correcto es el que el cuaderno calcula, y el caso
sirve de advertencia sobre redondear una sola vez y al final, que es
lo que la sección 4.4.2 del libro exige.

### 7.2 El Teorema 4.3, dos métricas que son una sola

El libro demuestra que si la desviación estándar de las observaciones
se calcula con divisor n, entonces el coeficiente de eficiencia y la
razón de error satisfacen una relación exacta. La consecuencia
práctica es que reportar ambas no aporta información adicional.

In [ ]:
for etiqueta, m in (("calibración", m_cal), ("validación", m_val)):
    izquierda = m["NSE"]
    derecha = 1.0 - m["RSR"]**2
    print(f"{etiqueta:12s} NSE = {izquierda:.6f} y "
          f"1 - RSR^2 = {derecha:.6f}, diferencia "
          f"{abs(izquierda - derecha):.2e}")
    assert abs(izquierda - derecha) < 1.0e-12

print("\nLa comprobación del libro sobre los valores redondeados:")
print(f"un coeficiente de 0.50 corresponde a una razón de "
      f"{np.sqrt(1 - 0.50):.3f}   (libro 0.707)")
print(f"un coeficiente de 0.75 corresponde a una razón de "
      f"{np.sqrt(1 - 0.75):.3f}   (libro 0.50)")

### 7.3 Lo que las métricas esconden

La Tabla 4.4 del libro recoge los umbrales orientativos publicados
para caudal simulado en paso mensual, con la advertencia de que se
establecieron para una variable, un paso de tiempo y una familia de
modelos concretos. El modelo cumple el criterio declarado, y aun así
los residuales revelan un defecto estructural que ninguna métrica
resume.

In [ ]:
def diagnostico_residuales(obs, sim) -> dict[str, float]:
    """Paridad, tendencia y estructura del residual, Listado 4.8."""
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    r = sim - obs
    pend_par = np.polyfit(obs, sim, 1)
    pend_res = np.polyfit(obs, r, 1)
    lag1 = np.corrcoef(r[:-1], r[1:])[0, 1]
    hetero = np.corrcoef(np.abs(r), obs)[0, 1]
    return dict(paridad=pend_par[0], tendencia=pend_res[0],
                autocorrelacion=lag1, heterocedasticidad=hetero)


def calificar(nse: float, rsr: float, pbias: float) -> str:
    """Umbrales orientativos de la Tabla 4.4 del libro."""
    if nse > 0.75 and rsr <= 0.50 and abs(pbias) < 10:
        return "muy bueno"
    if nse > 0.65 and rsr <= 0.60 and abs(pbias) < 15:
        return "bueno"
    if nse > 0.50 and rsr <= 0.70 and abs(pbias) < 25:
        return "satisfactorio"
    return "insatisfactorio"


diag_val = diagnostico_residuales(validacion["Q_obs_m3_s"],
                                  validacion["Q_sim_m3_s"])
for nombre, valor in diag_val.items():
    print(f"{nombre:20s} {valor: .3f}")
PUBLICADO_DIAG = {"paridad": 0.663, "tendencia": -0.337,
                  "autocorrelacion": 0.456,
                  "heterocedasticidad": 0.726}
print("\nEl libro publica 0.663, -0.337, 0.456 y 0.726.")
for nombre, libro_valor in PUBLICADO_DIAG.items():
    assert abs(diag_val[nombre] - libro_valor) <= 1.0e-3, nombre
print("La correlación entre la magnitud del residual y lo observado vale")
print(f"{diag_val['heterocedasticidad']:.5f}, que a tres decimales da")
print("0.725 y no el 0.726 impreso, otra vez por doble redondeo.")

cumple = (m_val["NSE"] > CRITERIO_ACEPTACION["NSE_minimo"]
          and abs(m_val["PBIAS"]) < CRITERIO_ACEPTACION["PBIAS_maximo"])
print(f"\nCalificación según la Tabla 4.4: "
      f"{calificar(m_val['NSE'], m_val['RSR'], m_val['PBIAS'])}")
print(f"Cumple el criterio declarado de antemano: {cumple}")
assert cumple

### 7.4 Las cuatro vistas de la Figura 4.7

La serie revela el desfase, el diagrama de paridad revela la
compresión del rango, la gráfica de residuales revela la tendencia y
el histograma revela el sesgo global. Una pendiente de paridad de
0.663 delata que el modelo satura la respuesta en crecidas, porque no
representa la escorrentía superficial que se activa con el suelo
saturado, proceso ausente en el periodo seco de calibración.

In [ ]:
obs_val = validacion["Q_obs_m3_s"].to_numpy()
sim_val = validacion["Q_sim_m3_s"].to_numpy()
res_val = sim_val - obs_val

fig, ejes = plt.subplots(2, 2, figsize=(10.5, 7.6))
(a1, a2), (a3, a4) = ejes
meses = np.arange(1, obs_val.size + 1)

a1.plot(meses, obs_val, "o-", color=PALETA["azul"], ms=3.4,
        label="observado")
a1.plot(meses, sim_val, "s--", color=PALETA["rojo"], ms=3.4,
        label="simulado")
a1.set_xlabel("Mes del periodo de validación")
a1.set_ylabel("Caudal (m3/s)")
a1.set_title("(a) serie del periodo reservado")
a1.legend(loc="upper left", fontsize=8)

limites = [0.0, 42.0]
a2.plot(limites, limites, "-", color=PALETA["gris"], lw=0.9,
        label="recta uno a uno")
a2.plot(calibracion["Q_obs_m3_s"], calibracion["Q_sim_m3_s"], "^",
        color=PALETA["verde"], ms=4.0, label="calibración")
a2.plot(obs_val, sim_val, "o", color=PALETA["rojo"], ms=4.0,
        label="validación")
recta = np.polyfit(obs_val, sim_val, 1)
a2.plot(limites, np.polyval(recta, limites), "--",
        color=PALETA["rojo"], lw=1.0)
a2.set_xlim(limites)
a2.set_ylim(limites)
a2.set_xlabel("Observado (m3/s)")
a2.set_ylabel("Simulado (m3/s)")
a2.set_title(f"(b) paridad, pendiente {recta[0]:.3f}")
a2.legend(loc="upper left", fontsize=7.5)

a3.axhline(0.0, color=PALETA["gris"], lw=0.9)
a3.plot(obs_val, res_val, "o", color=PALETA["rojo"], ms=4.0)
tendencia = np.polyfit(obs_val, res_val, 1)
malla_obs = np.linspace(obs_val.min(), obs_val.max(), 50)
a3.plot(malla_obs, np.polyval(tendencia, malla_obs), "--",
        color=PALETA["azul"], lw=1.1)
a3.set_xlabel("Observado (m3/s)")
a3.set_ylabel("Residual (m3/s)")
a3.set_title(f"(c) tendencia {tendencia[0]:.3f}")

a4.hist(res_val, bins=8, color=PALETA["azul"], alpha=0.75,
        edgecolor="white")
a4.axvline(0.0, color=PALETA["gris"], lw=0.9)
a4.axvline(res_val.mean(), color=PALETA["rojo"], lw=1.2, ls="--")
a4.set_xlabel("Residual (m3/s)")
a4.set_ylabel("Número de meses")
a4.set_title(f"(d) residual medio {res_val.mean():.2f} m3/s")

fig.tight_layout()
plt.show()

La conclusión defendible del Ejemplo 4.5 es que el modelo cumple el
criterio declarado y aun así no debe usarse para dimensionar una obra
de excedencias. Aceptarlo con dominio de validez restringido a
caudales inferiores a 20 m3/s es lo que corresponde, y ese es
precisamente el tipo de afirmación que la Unidad 4 pide escribir.

## 8. Ejercicios guiados

Las celdas siguientes están incompletas y llevan la marca
`# COMPLETE:`. Cada una arranca con un valor de partida
evidentemente incorrecto para que el cuaderno siga ejecutándose. Al
resolverlas, cambie `REVISAR = True` en la celda de configuración
para que las verificaciones se conviertan en comprobaciones estrictas.

### Ejercicio 1. Régimen asintótico

El Algoritmo 4.1 del libro avisa cuando dos estimaciones consecutivas
del orden difieren en más de 0.05, porque entonces no se alcanzó el
régimen asintótico. Escriba la función que devuelve el orden y esa
advertencia.

In [ ]:
def orden_con_diagnostico(h, e, tolerancia=0.05):
    """Orden por pares, pendiente global y aviso de régimen asintótico."""
    p_par, p_reg = orden_observado(h, e)
    # COMPLETE: asigne a asintotico el valor True cuando las dos
    # últimas estimaciones por pares difieran en menos de la
    # tolerancia, y False en caso contrario.
    asintotico = None
    return p_par, p_reg, asintotico


_, _, asintotico_aleta = orden_con_diagnostico(pasos, err_fantasma)
_, _, asintotico_corto = orden_con_diagnostico(pasos[:3], err_fantasma[:3])

In [ ]:
# Verificación del ejercicio 1.
verificar("régimen asintótico con seis mallas", asintotico_aleta, 1.0)
print("Con solo tres mallas el diagnóstico da:", asintotico_corto)

### Ejercicio 2. Problema 4-7 del libro

Con los valores 18.42, 18.61 y 18.68 de una magnitud integral en
mallas de razón 2, estime el orden observado, el valor extrapolado y
el índice de convergencia de malla. La función `richardson` ya está
definida en la sección 3.

In [ ]:
# COMPLETE: llame a richardson con los tres valores del problema 4-7,
# en el orden malla gruesa, intermedia y fina, y guarde el resultado
# en p_47, ext_47 y gci_47.
p_47, ext_47, gci_47 = None, None, None

In [ ]:
# Verificación del ejercicio 2.
verificar("orden observado del problema 4-7", p_47, 1.4406, tol=1.0e-3)
verificar("valor extrapolado del problema 4-7", ext_47, 18.7208, tol=1.0e-4)
verificar("índice de convergencia de malla", gci_47, 2.7324e-3, tol=1.0e-3)
if p_47 is not None:
    print(f"\nEl orden observado {p_47:.3f} no llega a dos, de modo que el")
    print("estudio no alcanzó el régimen asintótico y conviene refinar más.")

### Ejercicio 3. Criterio de información corregido

El problema 4-14 compara dos modelos que ajustan los mismos quince
datos con sumas de cuadrados de 2.4e-3 y 1.9e-3, con dos y cuatro
parámetros. Calcule la diferencia del criterio corregido de la
Ecuación 4.10 y decida cuál se prefiere. Recuerde que el número de
parámetros de la fórmula incluye la escala del error.

In [ ]:
# COMPLETE: use aic_corregido con n igual a 15 y con el número de
# parámetros aumentado en uno, y guarde la diferencia del modelo de
# cuatro parámetros menos el de dos en delta_aicc.
aicc_dos, aicc_cuatro = None, None
delta_aicc = None

In [ ]:
# Verificación del ejercicio 3.
verificar("criterio del modelo de dos parámetros", aicc_dos, -122.9232)
verificar("criterio del modelo de cuatro parámetros", aicc_cuatro, -117.9426)
verificar("diferencia entre criterios", delta_aicc, 4.9806)
if delta_aicc is not None:
    mejor = "el de dos parámetros" if delta_aicc > 0 else "el de cuatro"
    print(f"\nSe prefiere {mejor}. La diferencia de "
          f"{abs(delta_aicc):.2f} no llega a diez, de modo que el peor")
    print("situado no queda descartado, solo desfavorecido.")

### Ejercicio 4. Partición cronológica frente a partición al azar

El problema 4-18 pide explicar por qué partir al azar una serie
temporal produce métricas optimistas. Compruébelo sobre los datos del
Ejemplo 4.5, calculando el coeficiente de eficiencia de una partición
al azar de la mitad de los meses y comparándolo con el de la
partición cronológica.

In [ ]:
generador_particion = np.random.default_rng(SEMILLA)
indices = generador_particion.permutation(len(caudal))
reservados = np.sort(indices[: len(caudal) // 2])
azar = caudal.iloc[reservados]

# COMPLETE: calcule el coeficiente de eficiencia sobre la partición
# al azar y guárdelo en nse_azar. Use la función metricas.
nse_azar = None

In [ ]:
# Verificación del ejercicio 4.
verificar("eficiencia con partición al azar", nse_azar, 0.8331, tol=1.0e-3)
if nse_azar is not None:
    print(f"\nPartición al azar     NSE = {nse_azar:.3f}")
    print(f"Partición cronológica NSE = {m_val['NSE']:.3f}")
    print("La primera mezcla meses de los dos periodos y hereda parte del")
    print("acuerdo de la calibración, de modo que exagera el desempeño.")

### Ejercicio 5. Un residual con estructura

La autocorrelación de retardo unitario detecta si los errores llegan
en rachas. Cuente cuántas rachas del mismo signo hay en los
residuales de validación y compárelas con las que cabría esperar de
residuales independientes, que serían alrededor de la mitad de las
parejas consecutivas.

In [ ]:
# COMPLETE: cuente en cuántas de las parejas consecutivas de res_val
# el signo se conserva, y guarde el conteo en parejas_mismo_signo.
parejas_mismo_signo = None

In [ ]:
# Verificación del ejercicio 5.
verificar("parejas consecutivas del mismo signo", parejas_mismo_signo, 16.0)
if parejas_mismo_signo is not None:
    esperadas = (res_val.size - 1) / 2
    print(f"\nParejas del mismo signo {parejas_mismo_signo} de "
          f"{res_val.size - 1}, frente a {esperadas:.1f} esperadas")
    print(f"si los residuales fueran independientes. La autocorrelación")
    print(f"de retardo unitario vale {diag_val['autocorrelacion']:.3f}.")

## 9. Problemas del capítulo

### Problema 4-6, resuelto

Un esquema de segundo orden entrega errores de 4.10e-3, 1.05e-3 y
2.64e-4 para pasos de 0.04, 0.02 y 0.01. Calcule el orden observado
por pares y diga si se alcanzó el régimen asintótico.

In [ ]:
h_46 = np.array([0.04, 0.02, 0.01])
e_46 = np.array([4.10e-3, 1.05e-3, 2.64e-4])
p_par_46, p_reg_46, asintotico_46 = orden_con_diagnostico(h_46, e_46)

print(f"Orden por pares      {p_par_46[0]:.3f} y {p_par_46[1]:.3f}")
print(f"Pendiente global     {p_reg_46:.3f}")
print(f"Diferencia entre las dos estimaciones "
      f"{abs(p_par_46[1] - p_par_46[0]):.3f}")
print(f"Régimen asintótico alcanzado: {asintotico_46}")
print("\nAmbas estimaciones rondan dos y difieren en menos de 0.05,")
print("de modo que el esquema se comporta como su orden formal anuncia.")

### Problema 4-10, andamiaje

Un ajuste de dos parámetros entrega errores estándar de 0.021 y 1.45
con correlación de 0.987. Interprete el resultado y proponga una
reparametrización.

El andamiaje siguiente construye la matriz de covarianza que
corresponde a esos datos y dibuja la elipse de confianza conjunta.
Compárela con la del ajuste de Monod, cuya correlación es 0.778, y
escriba en una celda de texto la interpretación que el libro pide.

In [ ]:
ee_410 = np.array([0.021, 1.45])
corr_410 = 0.987
cov_410 = np.array([[ee_410[0]**2, corr_410 * ee_410[0] * ee_410[1]],
                    [corr_410 * ee_410[0] * ee_410[1], ee_410[1]**2]])

valores_propios = np.linalg.eigvalsh(cov_410)
razon_ejes = np.sqrt(valores_propios.max() / valores_propios.min())
print(f"Razón entre los semiejes de la elipse {razon_ejes:.1f}")

cov_monod = cov
razon_monod = np.sqrt(np.linalg.eigvalsh(cov_monod).max()
                      / np.linalg.eigvalsh(cov_monod).min())
print(f"La misma razón en el ajuste de Monod   {razon_monod:.1f}")
print("\nUna elipse tan alargada significa que los datos determinan bien")
print("una combinación de los dos parámetros y muy mal la otra. La vía es")
print("reparametrizar con esa combinación, como en la sección 6.4.")

### Problema 4-16, andamiaje

Una validación entrega error medio de -0.02 grados Celsius y raíz del
error cuadrático medio de 2.8 grados Celsius. Explique cómo coexisten
y qué gráfica revelaría lo que ocultan.

Construya una serie sintética con esas dos métricas, con la semilla
del curso, y dibújela para responder.

In [ ]:
generador_416 = np.random.default_rng(SEMILLA)
observado_416 = np.linspace(18.0, 34.0, 60)
residual_416 = generador_416.normal(0.0, 2.8, observado_416.size)
residual_416 = residual_416 - residual_416.mean() - 0.02
residual_416 *= 2.8 / np.sqrt(np.mean(residual_416**2))
residual_416 = residual_416 - residual_416.mean() - 0.02
simulado_416 = observado_416 + residual_416
m_416 = metricas(observado_416, simulado_416)

print(f"Error medio {m_416['ME']:.2f} grados Celsius y raíz del error "
      f"cuadrático medio {m_416['RMSE']:.2f} grados Celsius")
print("Los errores de signo opuesto se cancelan en el promedio, tal como")
print("advierte la Tabla 4.3 del libro para la métrica de sesgo.")

## Cierre

### Lo que debe saber hacer al terminar

- Escribir un caso de referencia con solución analítica y medir el error en norma del máximo.
- Estimar el orden observado por pares y por regresión, y decidir si se alcanzó el régimen asintótico.
- Aplicar la extrapolación de Richardson y reportar el índice de convergencia de malla como banda de incertidumbre numérica.
- Construir una solución fabricada con SymPy y verificar con ella un operador con coeficiente variable.
- Calibrar un modelo no lineal, reportar errores estándar, intervalos y correlación, y contrastarlos con bootstrap.
- Diagnosticar identificabilidad práctica con el número de condición y con el índice de colinealidad.
- Validar con datos independientes y examinar paridad, tendencia, autocorrelación y heterocedasticidad de los residuales.

### Qué revisar en el libro si algo no salió

- Si el orden observado no converge, revise el Teorema 4.1 y el Algoritmo 4.1 del libro.
- Si el término fuente de la solución fabricada no coincide, compare con la Ecuación 4.6 y con el Listado 4.1.
- Si los intervalos de los parámetros no coinciden, revise el Teorema 4.2 y el Listado 4.4.
- Si las métricas de validación no coinciden, revise el Listado 4.7 y la Tabla 4.3, que dice qué mide cada una y cuándo engaña.
- Si el diagnóstico de residuales le resulta ambiguo, revise la sección 4.3.2 y el Algoritmo 4.3.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente de programación.
Todo resultado numérico que aparece aquí se verifica contra el valor
que el libro publica, contra una solución analítica o contra un caso
límite, según recuerda la sección 4.6 del libro. La responsabilidad
del contenido no se transfiere al asistente.